# Code_6 — Human Evaluation, Final Tables and Figures

**GPU: no**

1. **Export** the annotation sheets for two independent raters (real LLM calls from Code_5).
2. **Import** the completed sheets (when they exist): Cohen's κ per question, disagreement list for
   adjudication, and system metrics against the adjudicated **human** labels.
3. **Build every paper table and figure** from the saved result files — no number is typed by hand.

There is **no simulated-rating fallback**: if the completed sheets are missing, the human-evaluation
parts are skipped and say so.

Human-evaluation files live in `runs/<run_id>/human_eval/`:

| File | Who fills it |
|---|---|
| `ANNOTATION_GUIDELINES.md` | read by both raters first |
| `practice_rater1.csv`, `practice_rater2.csv` | both raters, then discuss and refine the guideline |
| `pass1_rater1.csv`, `pass1_rater2.csv` | each rater alone: is the call wrong, which errors, the correct call |
| `pass2_rater1.csv`, `pass2_rater2.csv` | each rater alone, after pass 1: is the corrected call valid / does it do what was asked |
| `adjudication.csv` | the third person, for the disagreements listed by this notebook |

In [ ]:
import os, sys, json, importlib, csv, random, copy
from collections import Counter, defaultdict

IN_COLAB = os.path.exists('/content')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/hallucination_project'
else:
    DRIVE_ROOT = os.environ.get('TOOLHALL_ROOT', os.path.abspath('hallucination_project'))
CODE_DIR = os.path.join(DRIVE_ROOT, 'code')
sys.path.insert(0, CODE_DIR)
import toolhall_core as th
importlib.reload(th)
import numpy as np

NOTEBOOK = 'Code_6'
CFG = th.load_config(os.path.join(CODE_DIR, 'config.json'))
CFG['drive_root'] = DRIVE_ROOT
P = th.Paths(CFG)
SEED = CFG['seed']
HE = os.path.join(P.run, 'human_eval')
FIG = os.path.join(P.run, 'figures')
TAB = os.path.join(P.run, 'tables')
for d in (HE, FIG, TAB):
    os.makedirs(d, exist_ok=True)


def load_opt(name, kind='run'):
    path = P.run_file(name) if kind == 'run' else P.data_file(name)
    try:
        return th.load_json(path, CFG, kind=kind)
    except FileNotFoundError:
        print(f"  (missing {name} -- the parts that need it are skipped)")
        return None


registry = th.Registry(th.load_json(P.data_file('registry.json'), CFG, kind='data'))
real = load_opt('real_eval.json', kind='data')
data_summary = load_opt('data_summary.json', kind='data')
examples_by_type = load_opt('examples_by_type.json', kind='data')
refm = load_opt('reference_metrics.json')
core = load_opt('core_results.json')
core_rec = load_opt('core_records.json')
single_bm = load_opt('benchmark_single.json', kind='data')
base = load_opt('baselines.json')
retr = load_opt('retrieval.json')
realres = load_opt('real_results.json')
realrec = load_opt('real_records.json')
selfchk = load_opt('selfcheck_results.json')
print(f"run_id={CFG['run_id']}")

## Part 1 — Export the annotation sheets

`human_eval.n_items` calls are sampled from the free-generation, pipeline-condition records, stratified
by model × tool group × whether the system flagged the call, so both flagged and unflagged calls and
every model are represented. Item IDs are random; the model name is **not** shown to raters.
Pass 1 never shows the system's correction (so raters judge the call on its own); pass 2 does.
The guideline and the item list are written once and reused, so re-running does not reshuffle.

In [ ]:
GUIDE = """# Annotation guidelines -- tool-call hallucination study

You will see a user request, the schema of the available function(s) and a tool call produced by a
language model. Judge the call ONLY against the request and the schema. Work alone; do not discuss
items with the other rater until both of you have finished.

## Pass 1 (sheet pass1_raterN.csv) -- do this first
* `call_is_wrong`: `yes` if the call would not correctly do what the user asked, else `no`.
* `error_types`: if wrong, one or more of the codes below, separated by `;`
  * WRONG_FUNCTION -- names a function that does not exist, or the wrong one of the available functions
  * EXTRA_PARAMETER -- passes a parameter the function does not declare
  * MISSING_PARAMETER -- leaves out a parameter that the request needs or the schema requires
  * WRONG_TYPE -- right parameter, value of the wrong type (e.g. "10" where a number is declared)
  * WRONG_VALUE -- right parameter and type, but the value does not match the request
  * WRONG_FORMAT -- not a single {"function": ..., "parameters": {...}} JSON object
  * OTHER -- wrong for a reason none of the above covers (explain in notes)
* `correct_call`: if wrong, write the call that SHOULD have been made, as JSON
  {"function": "...", "parameters": {...}}. Leave empty if the call is right.

## Pass 2 (sheet pass2_raterN.csv) -- only after finishing pass 1
* `corrected_call_valid`: `yes` if the corrected call is a well-formed call that matches the schema.
* `corrected_call_intent`: `yes` / `partly` / `no` -- does the corrected call do what the user asked?

## Edge cases (agree on more during the practice round and add them here)
* An empty string "" for an optional parameter means "not set"; it is not an error.
* Numbers written as strings ("5") count as WRONG_TYPE only if the schema declares a number.
* A value that is reasonable but not exactly what the user typed (e.g. "NYC" for "New York") is NOT wrong
  unless the API would clearly misread it.
* If the request needs two calls, judge whether this call is a correct FIRST step.
"""

items_file = os.path.join(HE, 'items.json')
pipe_free = [r for r in (realrec or []) if r['condition'] == 'pipeline' and r['mode'] == 'free']
if pipe_free and not os.path.exists(items_file):
    rng = random.Random(SEED)
    strata = defaultdict(list)
    for r in pipe_free:
        strata[(r['model'], r['tool_group'], r['flagged'])].append(r)
    n_total = CFG['human_eval']['n_items'] + CFG['human_eval']['n_practice']
    keys = sorted(strata)
    per = max(1, n_total // len(keys))
    chosen = []
    for k in keys:
        L = sorted(strata[k], key=lambda r: r['id'])
        rng.shuffle(L)
        chosen += L[:per]
    rest = [r for r in pipe_free if r not in chosen]
    rng.shuffle(rest)
    chosen = (chosen + rest)[:n_total]
    rng.shuffle(chosen)
    gold_by_id = {r['id']: r for r in real}
    items = []
    for k, r in enumerate(chosen):
        g = gold_by_id[r['id']]
        items.append({'item_id': f"H{rng.randrange(10**6):06d}", 'set': 'practice' if k < CFG['human_eval']['n_practice'] else 'main',
                      'record_id': r['id'], 'model': r['model'], 'tool_group': r['tool_group'],
                      'query': g['query'],
                      'schema': [th.schema_for_prompt(registry, f) for f in g['tool_functions'] if registry.resolve(f)],
                      'generated_call': r['raw_output'], 'corrected_call': json.dumps(r['final_call'], ensure_ascii=False, default=str),
                      'system_flagged': r['flagged'], 'auto_labels': r['auto_labels'], 'gold_call': g['gt_call']})
    json.dump(items, open(items_file, 'w', encoding='utf-8'), indent=1, ensure_ascii=False)
    with open(os.path.join(HE, 'ANNOTATION_GUIDELINES.md'), 'w', encoding='utf-8') as f:
        f.write(GUIDE)

    def write_sheet(path, rows, cols):
        with open(path, 'w', newline='', encoding='utf-8-sig') as f:
            w = csv.DictWriter(f, fieldnames=cols)
            w.writeheader()
            for row in rows:
                w.writerow(row)

    for rater in (1, 2):
        for set_name, prefix in (('practice', 'practice'), ('main', 'pass1')):
            rows = [{'item_id': it['item_id'], 'user_request': it['query'],
                     'function_schemas': json.dumps(it['schema'], ensure_ascii=False)[:4000],
                     'tool_call': it['generated_call'], 'call_is_wrong': '', 'error_types': '',
                     'correct_call': '', 'notes': ''} for it in items if it['set'] == set_name]
            write_sheet(os.path.join(HE, f"{prefix}_rater{rater}.csv"), rows,
                        ['item_id', 'user_request', 'function_schemas', 'tool_call', 'call_is_wrong',
                         'error_types', 'correct_call', 'notes'])
        rows = [{'item_id': it['item_id'], 'user_request': it['query'],
                 'function_schemas': json.dumps(it['schema'], ensure_ascii=False)[:4000],
                 'tool_call': it['generated_call'], 'corrected_call': it['corrected_call'],
                 'corrected_call_valid': '', 'corrected_call_intent': '', 'notes': ''}
                for it in items if it['set'] == 'main']
        write_sheet(os.path.join(HE, f"pass2_rater{rater}.csv"), rows,
                    ['item_id', 'user_request', 'function_schemas', 'tool_call', 'corrected_call',
                     'corrected_call_valid', 'corrected_call_intent', 'notes'])
    print(f"exported {len(items)} items ({CFG['human_eval']['n_practice']} practice) to {HE}")
elif os.path.exists(items_file):
    items = json.load(open(items_file, encoding='utf-8'))
    print(f"annotation items already exported ({len(items)}) -- not reshuffled")
else:
    items = []
    print("no real-call records yet (run Code_5 first) -- nothing to export")

## Part 2 — Import completed sheets, agreement, adjudication, metrics against human labels

Completed sheets must keep the same file names and columns. κ is computed on the main set only.

In [ ]:
def read_sheet(name):
    path = os.path.join(HE, name)
    if not os.path.exists(path):
        return None
    with open(path, encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    filled = [r for r in rows if any((r.get(c) or '').strip() for c in
                                     ('call_is_wrong', 'corrected_call_valid'))]
    return {r['item_id']: r for r in filled} if filled else None


def yn(v):
    v = (v or '').strip().lower()
    return {'yes': 1, 'y': 1, 'no': 0, 'n': 0}.get(v)


def cohen_kappa(a, b):
    cats = sorted(set(a) | set(b))
    n = len(a)
    if n == 0:
        return None
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(c) / n) * (b.count(c) / n) for c in cats)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


def weighted_kappa(a, b, order):
    idx = {c: i for i, c in enumerate(order)}
    a = [idx[x] for x in a]
    b = [idx[x] for x in b]
    k = len(order)
    n = len(a)
    if n == 0 or k < 2:
        return None
    w = lambda i, j: abs(i - j) / (k - 1)
    obs = sum(w(x, y) for x, y in zip(a, b)) / n
    pa = [a.count(i) / n for i in range(k)]
    pb = [b.count(i) / n for i in range(k)]
    exp = sum(pa[i] * pb[j] * w(i, j) for i in range(k) for j in range(k))
    return 1 - obs / exp if exp > 0 else 1.0


HUMAN = {'status': 'not available'}
r1p1, r2p1 = read_sheet('pass1_rater1.csv'), read_sheet('pass1_rater2.csv')
r1p2, r2p2 = read_sheet('pass2_rater1.csv'), read_sheet('pass2_rater2.csv')
if r1p1 and r2p1:
    common = sorted(set(r1p1) & set(r2p1))
    a = [yn(r1p1[i]['call_is_wrong']) for i in common]
    b = [yn(r2p1[i]['call_is_wrong']) for i in common]
    ok = [k for k, (x, y) in enumerate(zip(a, b)) if x is not None and y is not None]
    HUMAN = {'status': 'computed from completed sheets', 'n_items_both_raters': len(common)}
    HUMAN['kappa_call_is_wrong'] = cohen_kappa([a[k] for k in ok], [b[k] for k in ok])
    HUMAN['agreement_call_is_wrong'] = float(np.mean([a[k] == b[k] for k in ok])) if ok else None
    types_k = {}
    for t in th.ERROR_TYPES + ['OTHER']:
        x = [int(t in (r1p1[i]['error_types'] or '').upper()) for i in common]
        y = [int(t in (r2p1[i]['error_types'] or '').upper()) for i in common]
        if sum(x) + sum(y) > 0:
            types_k[t] = {'kappa': cohen_kappa(x, y), 'rater1_count': sum(x), 'rater2_count': sum(y)}
    HUMAN['kappa_error_type'] = types_k
    if r1p2 and r2p2:
        c2 = sorted(set(r1p2) & set(r2p2))
        v1 = [yn(r1p2[i]['corrected_call_valid']) for i in c2]
        v2 = [yn(r2p2[i]['corrected_call_valid']) for i in c2]
        okv = [k for k in range(len(c2)) if v1[k] is not None and v2[k] is not None]
        HUMAN['kappa_corrected_valid'] = cohen_kappa([v1[k] for k in okv], [v2[k] for k in okv])
        order = ['no', 'partly', 'yes']
        i1 = [(r1p2[i]['corrected_call_intent'] or '').strip().lower() for i in c2]
        i2 = [(r2p2[i]['corrected_call_intent'] or '').strip().lower() for i in c2]
        oki = [k for k in range(len(c2)) if i1[k] in order and i2[k] in order]
        HUMAN['kappa_intent'] = cohen_kappa([i1[k] for k in oki], [i2[k] for k in oki])
        HUMAN['weighted_kappa_intent'] = weighted_kappa([i1[k] for k in oki], [i2[k] for k in oki], order)
    # disagreements -> adjudication sheet (written once)
    adj_path = os.path.join(HE, 'adjudication.csv')
    dis = [i for i in common if yn(r1p1[i]['call_is_wrong']) != yn(r2p1[i]['call_is_wrong'])
           or set((r1p1[i]['error_types'] or '').upper().replace(' ', '').split(';')) !=
           set((r2p1[i]['error_types'] or '').upper().replace(' ', '').split(';'))]
    HUMAN['n_disagreements'] = len(dis)
    if not os.path.exists(adj_path):
        with open(adj_path, 'w', newline='', encoding='utf-8-sig') as f:
            w = csv.writer(f)
            w.writerow(['item_id', 'rater1_wrong', 'rater1_types', 'rater2_wrong', 'rater2_types',
                        'final_call_is_wrong', 'final_error_types', 'final_correct_call'])
            for i in dis:
                w.writerow([i, r1p1[i]['call_is_wrong'], r1p1[i]['error_types'], r2p1[i]['call_is_wrong'],
                            r2p1[i]['error_types'], '', '', ''])
        print(f"wrote {len(dis)} disagreements to adjudication.csv")
    # human gold = agreement, or adjudicated decision
    adj = {}
    if os.path.exists(adj_path):
        with open(adj_path, encoding='utf-8-sig') as f:
            adj = {r['item_id']: r for r in csv.DictReader(f) if (r.get('final_call_is_wrong') or '').strip()}
    item_by = {it['item_id']: it for it in items}
    human_wrong = {}
    for i in common:
        if i in adj:
            human_wrong[i] = yn(adj[i]['final_call_is_wrong'])
        elif i not in dis:
            human_wrong[i] = yn(r1p1[i]['call_is_wrong'])
    usable = [i for i, v in human_wrong.items() if v is not None and i in item_by]
    HUMAN['n_with_human_gold'] = len(usable)
    HUMAN['n_disagreements_unresolved'] = len([i for i in dis if i not in adj])
    if usable:
        y = [human_wrong[i] for i in usable]
        sys_flag = [int(item_by[i]['system_flagged']) for i in usable]
        auto = [int(bool(item_by[i]['auto_labels'])) for i in usable]
        HUMAN['system_detection_vs_human'] = th.binary_metrics(y, sys_flag)
        if selfchk:
            pr = selfchk['per_record']
            key = lambda i: f"{item_by[i]['model']}|{item_by[i]['record_id']}"
            both = [i for i in usable if key(i) in pr]
            if both:
                yb = [human_wrong[i] for i in both]
                HUMAN['selfcheck_vs_human'] = {
                    'n': len(both),
                    'selfcheck': th.binary_metrics(yb, [pr[key(i)]['flag'] for i in both]),
                    'schema_only_same_items': th.binary_metrics(yb, [int(item_by[i]['system_flagged']) for i in both]),
                    'union_same_items': th.binary_metrics(yb, [int(pr[key(i)]['flag'] or item_by[i]['system_flagged'])
                                                               for i in both])}
        HUMAN['auto_labels_vs_human_kappa'] = cohen_kappa(y, auto)
        HUMAN['auto_labels_vs_human_agreement'] = float(np.mean([a_ == b_ for a_, b_ in zip(y, auto)]))
    if r1p2 and r2p2:
        both_yes = [i for i in sorted(set(r1p2) & set(r2p2))
                    if (r1p2[i]['corrected_call_intent'] or '').strip().lower() == 'yes'
                    and (r2p2[i]['corrected_call_intent'] or '').strip().lower() == 'yes']
        HUMAN['corrected_intent_yes_both_raters'] = len(both_yes) / max(1, len(set(r1p2) & set(r2p2)))
    print(json.dumps(th.strip_ci(HUMAN), indent=1))
else:
    print("Human evaluation: completed pass-1 sheets from BOTH raters not found in", HE)
    print("-> fill pass1_rater1.csv and pass1_rater2.csv (and pass 2), upload them there and re-run this notebook.")
    print("   No numbers are produced without real ratings.")

## Part 3 — Final tables (CSV + LaTeX) and figures, generated from the result files

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

TABLES = {}


def save_table(name, header, rows, caption):
    TABLES[name] = {'header': header, 'rows': rows, 'caption': caption}
    with open(os.path.join(TAB, name + '.csv'), 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(header)
        w.writerows(rows)
    fmt = lambda v: f"{v:.3f}" if isinstance(v, float) else str(v).replace('_', r'\_').replace('%', r'\%')
    with open(os.path.join(TAB, name + '.tex'), 'w', encoding='utf-8') as f:
        f.write('\\begin{table}[t]\n\\centering\n\\caption{' + caption + '}\n\\label{tab:' + name + '}\n')
        f.write('\\begin{tabular}{' + 'l' + 'r' * (len(header) - 1) + '}\n\\hline\n')
        f.write(' & '.join(fmt(h) for h in header) + ' \\\\\n\\hline\n')
        for r in rows:
            f.write(' & '.join(fmt(v) for v in r) + ' \\\\\n')
        f.write('\\hline\n\\end{tabular}\n\\end{table}\n')


def savefig(name):
    plt.tight_layout()
    for ext in ('png', 'pdf'):
        plt.savefig(os.path.join(FIG, f"{name}.{ext}"), dpi=200)
    plt.close()


if data_summary:
    s = data_summary['single']['by_label']
    save_table('benchmark_composition', ['label', 'examples'],
               [[k, v] for k, v in sorted(s.items())] + [['total', sum(s.values())]],
               'Single-error benchmark composition')
    sp = data_summary['splits']
    save_table('splits', ['split', 'tune', 'test', 'functions in both'],
               [[k, v['n_tune'], v['n_test'], v['functions_in_both']] for k, v in sp.items()],
               'Random and function-disjoint splits')

if examples_by_type:
    save_table('examples_by_type', ['label', 'reference call', 'generated call'],
               [[k, json.dumps(v['reference_call'], ensure_ascii=False)[:90],
                 json.dumps(v['generated_call'], ensure_ascii=False)[:90]] for k, v in examples_by_type.items()],
               'One example per error type (before / after injection)')

if core:
    for split_name in ('random', 'function_disjoint'):
        det = core['main'][split_name]['detection']
        cor = core['main'][split_name]['correction']
        mc = det['classification']
        save_table(f'classwise_{split_name}', ['class', 'precision', 'recall', 'F1', 'support'],
                   [[l, mc['per_class'][l]['precision'], mc['per_class'][l]['recall'], mc['per_class'][l]['f1'],
                     mc['per_class'][l]['support']] for l in th.ALL_LABELS]
                   + [['macro', '', '', mc['macro_f1'], '']],
                   f'Class-wise precision, recall and F1 ({split_name.replace("_", "-")} test split)')
        cm = np.array(mc['confusion_matrix'])
        plt.figure(figsize=(6.4, 5.4))
        plt.imshow(cm, cmap='Blues')
        plt.xticks(range(len(mc['labels'])), [l.replace('_', '\n') for l in mc['labels']], fontsize=7)
        plt.yticks(range(len(mc['labels'])), mc['labels'], fontsize=7)
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, cm[i, j], ha='center', va='center', fontsize=7,
                         color='white' if cm[i, j] > cm.max() / 2 else 'black')
        plt.xlabel('predicted')
        plt.ylabel('true')
        plt.title(f'Confusion matrix — {split_name.replace("_", "-")} test split')
        savefig(f'confusion_{split_name}')
    rows = []
    for split_name in ('random', 'function_disjoint'):
        det, cor = core['main'][split_name]['detection'], core['main'][split_name]['correction']
        rows.append([split_name, det['ci']['balanced_accuracy']['point'], det['binary']['f1'],
                     det['binary']['fpr'], det['binary']['fnr'], det['classification']['macro_f1'],
                     cor['final_schema_valid_rate_hallucinated'], cor['final_exact_rate_all'],
                     cor['final_exact_rate_hallucinated']])
    save_table('main_results', ['split', 'bal. acc', 'F1', 'FPR', 'FNR', 'macro-F1', 'schema-valid (H)',
                                'exact (all)', 'exact (H)'], rows,
               'Main results: detection, classification, correction, intent proxy (H = hallucinated calls)')
    # ablation
    for split_name in ('random', 'function_disjoint'):
        A = core['ablation'][split_name]
        rows = [[n, d['summary']['det_f1'], d['summary']['exact_all'],
                 (d['vs_full']['diff_ci']['diff'] if d['vs_full'] else 0.0),
                 (d['vs_full']['mcnemar']['p_value'] if d['vs_full'] else '')] for n, d in A.items()]
        save_table(f'ablation_{split_name}', ['variant', 'det. F1', 'exact', 'Δ exact', 'McNemar p'], rows,
                   f'Ablation ({split_name.replace("_", "-")} test split)')
        names = [r[0] for r in rows if r[0] != 'full']
        deltas = [r[3] for r in rows if r[0] != 'full']
        plt.figure(figsize=(7, 0.32 * len(names) + 1))
        plt.barh(names, deltas, color=['#2a7' if d > 0 else '#c44' for d in deltas])
        plt.axvline(0, color='k', lw=0.8)
        plt.xlabel('Δ exact match vs full system')
        plt.title(f'Ablation — {split_name.replace("_", "-")}')
        savefig(f'ablation_{split_name}')
    # schema noise
    plt.figure(figsize=(6, 4))
    for split_name, style in (('random', '-o'), ('function_disjoint', '--s')):
        ms = core['schema_noise'][split_name]['mean_std']
        rates = sorted(float(k) for k in ms)
        for metric, lab in (('det_f1', 'detection F1'), ('exact_all', 'exact match')):
            m = [ms[str(r) if str(r) in ms else repr(r)][metric][0] for r in rates]
            e = [ms[str(r) if str(r) in ms else repr(r)][metric][1] for r in rates]
            plt.errorbar([100 * r for r in rates], m, yerr=e, fmt=style, capsize=3,
                         label=f"{lab} ({split_name.replace('_', '-')})")
    plt.xlabel('% of schema declarations corrupted')
    plt.ylabel('score')
    plt.legend(fontsize=7)
    plt.title('Robustness to schema errors')
    savefig('schema_noise')
    # multi-error
    M = core['multi_error']['by_k']
    save_table('multi_error', ['errors per call', 'n', 'detected', 'schema-valid', 'exact', 'set acc (trace)'],
               [[k, d['n'], d['detected'], d['schema_valid'], d['exact'], d['correction_trace']['exact_set_accuracy']]
                for k, d in M.items()], 'Calls with several simultaneous errors')
    plt.figure(figsize=(5, 3.6))
    ks = list(M.keys())
    plt.plot(ks, [M[k]['exact'] for k in ks], '-o', label='exact match')
    plt.plot(ks, [M[k]['schema_valid'] for k in ks], '-s', label='schema-valid')
    plt.plot(ks, [M[k]['correction_trace']['exact_set_accuracy'] for k in ks], '-^', label='error set identified')
    plt.xlabel('errors per call')
    plt.legend(fontsize=8)
    plt.title('Multi-error calls')
    savefig('multi_error')
    # type-trust sweep
    for split_name in ('random', 'function_disjoint'):
        T = core['type_checking'][split_name]
        taus = [k for k in T if k.startswith('adaptive')]
        plt.figure(figsize=(5, 3.6))
        plt.plot([float(k.split('=')[1]) for k in taus], [T[k]['summary']['fpr'] for k in taus], '-o', label='FPR (clean)')
        plt.plot([float(k.split('=')[1]) for k in taus], [T[k]['wrong_type_recall'] for k in taus], '-s',
                 label='WRONG_TYPE recall')
        for mode, mk in (('strict', 'x'), ('skiplist', 'd')):
            plt.scatter([1.02], [T[mode]['summary']['fpr']], marker=mk, label=f'{mode} FPR')
        plt.xlabel('trust threshold τ')
        plt.legend(fontsize=7)
        plt.title(f'Adaptive type checking — {split_name.replace("_", "-")}')
        savefig(f'type_trust_{split_name}')
        save_table(f'type_strategies_{split_name}', ['strategy', 'FPR', 'WRONG_TYPE recall', 'det. F1', 'exact'],
                   [[k, v['summary']['fpr'], v['wrong_type_recall'], v['summary']['det_f1'], v['summary']['exact_all']]
                    for k, v in T.items() if isinstance(v, dict) and 'summary' in v],
                   f'Type-checking strategies ({split_name.replace("_", "-")})')
    # WRONG_VALUE settings
    rows = []
    for split_name in ('random', 'function_disjoint'):
        for name, d in core['wrong_value'][split_name].items():
            rows.append([f"{split_name}: {name}", d['wv_detected'], d['wv_exact'], d['clean_fpr'], d['all_exact']])
    if retr:
        for setting, dd in retr.items():
            if setting.startswith('_'):
                continue
            for key, d in dd.items():
                rows.append([f"{key.split('/')[0]}: {setting} ({key.split('/')[1]})", '', d['wrong_value_exact_after'],
                             '', d['end_to_end_exact_after']])
    save_table('wrong_value_settings', ['setting', 'WV detected', 'WV exact', 'clean FPR', 'all exact'], rows,
               'WRONG_VALUE: schema only vs schema + query vs + retrieval')

if core and refm and base:
    for split_name in ('random', 'function_disjoint'):
        rows = [[r[0], r[1], r[2], r[3], r[4], r[5], r[6] if r[6] is not None else '']
                for r in base['comparison'][split_name]['detection_rows']]
        save_table(f'baselines_detection_{split_name}',
                   ['method', 'deployable', 'bal. acc', 'F1', 'FPR', 'FNR', 'macro-F1'], rows,
                   f'Detection: schema-only vs baselines ({split_name.replace("_", "-")} test split)')
        save_table(f'baselines_correction_{split_name}', ['method', 'exact match'],
                   base['comparison'][split_name]['correction_rows'],
                   f'Correction: schema-only vs baselines ({split_name.replace("_", "-")} test split)')

if realres:
    rows = []
    for key, d in realres['per_model'].items():
        m, mode, grp = key.split('|')
        rows.append([m.split('/')[-1], mode, grp, d['n'], d['parse_rate'], d['raw_exact'], d['detection']['f1'],
                     d['error_type_macro_f1'], d['final_schema_valid'], d['final_exact'], d['final_lenient']])
    save_table('cross_model', ['model', 'decoding', 'tools', 'n', 'parse', 'raw exact', 'det. F1', 'type F1',
                               'final schema-valid', 'final exact', 'final lenient'], rows,
               'Real LLM tool calls: raw vs after the schema-only pipeline, known vs unseen tools')

if selfchk:
    rows = []
    for m, d in selfchk['by_model'].items():
        for name, label in (('selfcheck', 'SelfCheckGPT-style'), ('schema_only', 'schema-only (ours)'),
                            ('union', 'ours OR SelfCheck')):
            b = d[name]
            rows.append([m.split('/')[-1], label, b['balanced_accuracy'], b['precision'], b['recall'], b['f1'], b['fpr']])
    save_table('selfcheck_baseline', ['model', 'method', 'bal. acc', 'precision', 'recall', 'F1', 'FPR'], rows,
               f"Published baseline: SelfCheckGPT-style sampling consistency ({selfchk['n_samples']} samples) "
               f"vs schema-only on real LLM calls")
    if HUMAN.get('selfcheck_vs_human'):
        h = HUMAN['selfcheck_vs_human']
        save_table('selfcheck_vs_human', ['method', 'n', 'bal. acc', 'precision', 'recall', 'F1', 'FPR'],
                   [[lab, h['n'], h[k]['balanced_accuracy'], h[k]['precision'], h[k]['recall'], h[k]['f1'], h[k]['fpr']]
                    for k, lab in (('selfcheck', 'SelfCheckGPT-style'), ('schema_only_same_items', 'schema-only (ours)'),
                                   ('union_same_items', 'ours OR SelfCheck'))],
                   'Detection against the human labels (same items)')

EXAMPLES = {}
if core_rec and single_bm:
    by_id = {e['id']: e for e in single_bm}
    rng_ex = random.Random(SEED)
    rows = []
    for lab in th.ALL_LABELS[1:]:
        recs = [r for r in core_rec['random'] if r['true_types'] == [lab]]
        ok = [r for r in recs if r['final_exact']]
        bad = [r for r in recs if not r['final_exact']]
        for kind, pool in (('success', ok), ('failure', bad)):
            if not pool:
                continue
            r = rng_ex.choice(sorted(pool, key=lambda x: x['id']))
            ex = by_id[r['id']]
            item = {'id': r['id'], 'true_type': lab, 'outcome': kind, 'category': th.failure_category(r),
                    'query': ex['query'][:300], 'generated': ex['generated_tool_call'],
                    'reference': ex['ground_truth_tool_call'], 'final': r['final_call'],
                    'detected_as': r['pred_types'], 'actions': r['actions']}
            EXAMPLES.setdefault('synthetic', []).append(item)
            rows.append([lab, kind, item['category'], json.dumps(ex['generated_tool_call'], ensure_ascii=False)[:80],
                         json.dumps(r['final_call'], ensure_ascii=False, default=str)[:80],
                         '; '.join(r['actions'])[:60] or '-'])
    save_table('correction_examples_synthetic',
               ['true error', 'outcome', 'category', 'generated call', 'system output', 'fixes applied'], rows,
               'Examples of successful and failed corrections (synthetic test calls)')
if realrec and real:
    gold_by = {r['id']: r for r in real}
    rng_ex = random.Random(SEED + 1)
    pf = [r for r in realrec if r['condition'] == 'pipeline' and r['mode'] == 'free']
    groups = {'corrected (wrong -> exact)': [r for r in pf if r['flagged'] and r['final_exact'] and not r['raw_exact']],
              'flagged, not fixed': [r for r in pf if r['flagged'] and not r['final_exact']],
              'missed (semantic error)': [r for r in pf if r['is_hallucinated'] and not r['flagged']],
              'correct, left untouched': [r for r in pf if not r['is_hallucinated'] and not r['flagged']]}
    rows = []
    for g, pool in groups.items():
        for r in sorted(rng_ex.sample(sorted(pool, key=lambda x: (x['model'], x['id'])), min(2, len(pool))),
                        key=lambda x: x['id']):
            item = {'id': r['id'], 'model': r['model'], 'group': g, 'query': gold_by[r['id']]['query'],
                    'generated': r['raw_output'][:400], 'final': r['final_call'], 'gold': gold_by[r['id']]['gt_call'],
                    'auto_labels': r['auto_labels']}
            EXAMPLES.setdefault('real', []).append(item)
            rows.append([g, r['model'].split('/')[-1], gold_by[r['id']]['query'][:70],
                         r['raw_output'][:70].replace(chr(10), " "),
                         json.dumps(r['final_call'], ensure_ascii=False, default=str)[:70],
                         json.dumps(gold_by[r['id']]['gt_call'], ensure_ascii=False)[:70]])
    save_table('correction_examples_real', ['case', 'model', 'request', 'model output', 'system output', 'gold call'],
               rows, 'Examples of successful and failed corrections on real LLM calls')
if EXAMPLES:
    with open(os.path.join(TAB, 'correction_examples_full.json'), 'w', encoding='utf-8') as f:
        json.dump(EXAMPLES, f, indent=1, ensure_ascii=False, default=str)
    print(f"correction examples: {len(EXAMPLES.get('synthetic', []))} synthetic, {len(EXAMPLES.get('real', []))} real")

if core:
    lat = core['runtime']
    rows = []
    for split_name in ('random', 'function_disjoint'):
        for comp, d in lat[split_name].items():
            rows.append([f"{split_name}: {comp}", d['mean_ms'], d['p50_ms'], d['p95_ms'], d['p99_ms']])
    if lat.get('matcher_only'):
        d = lat['matcher_only']
        rows.append(['function matcher (per WRONG_FUNCTION call)', d['mean_ms'], d['p50_ms'], d['p95_ms'], d['p99_ms']])
    save_table('runtime', ['component', 'mean ms', 'p50 ms', 'p95 ms', 'p99 ms'], rows,
               f"Latency per call ({lat['hardware'].get('cpu_model') or lat['hardware'].get('processor')})")

if HUMAN.get('status', '').startswith('computed'):
    save_table('human_agreement', ['question', 'Cohen κ'],
               [['call is wrong', HUMAN.get('kappa_call_is_wrong')],
                ['corrected call valid', HUMAN.get('kappa_corrected_valid')],
                ['corrected call satisfies intent', HUMAN.get('kappa_intent')]]
               + [[f"error type {t}", d['kappa']] for t, d in HUMAN.get('kappa_error_type', {}).items()],
               'Inter-rater agreement')

print('tables :', sorted(os.listdir(TAB)))
print('figures:', sorted(f for f in os.listdir(FIG) if f.endswith('.png')))

## Part 4 — `paper_numbers.json`: the only place the paper takes numbers from

In [ ]:
PN = {'run_id': CFG['run_id'], 'data_version': CFG['data_version'], 'human_evaluation': HUMAN}
if data_summary:
    PN['benchmark'] = {'single': data_summary['single']['by_label'], 'multi_by_k': {k: v['made'] for k, v in
                                                                                  data_summary['multi']['by_k'].items()},
                       'registry_functions': sum(data_summary['registry']['functions_by_source'].values()),
                       'splits': data_summary['splits']}
if core:
    PN['main'] = {s: {'detection_balanced_accuracy': core['main'][s]['detection']['ci']['balanced_accuracy'],
                      'detection_f1': core['main'][s]['detection']['ci']['f1'],
                      'fpr': core['main'][s]['detection']['binary']['fpr'],
                      'classification_macro_f1': core['main'][s]['detection']['classification']['ci_macro_f1'],
                      'exact_all': core['main'][s]['correction']['ci']['final_exact_rate_all'],
                      'exact_hallucinated': core['main'][s]['correction']['ci']['final_exact_rate_hallucinated'],
                      'schema_valid_hallucinated': core['main'][s]['correction']['final_schema_valid_rate_hallucinated']}
                  for s in ('random', 'function_disjoint')}
    PN['tuned'] = core['tuned']
if realres:
    PN['real_llm'] = {k: {'raw_exact': v['raw_exact'], 'final_exact': v['final_exact'],
                          'detection_f1': v['detection']['f1']} for k, v in realres['per_model'].items()}
if selfchk:
    PN['selfcheck_baseline'] = {m: {k: d[k]['balanced_accuracy'] for k in ('selfcheck', 'schema_only', 'union')}
                                for m, d in selfchk['by_model'].items()}
th.save_json(PN, P.run_file('paper_numbers.json'), CFG, NOTEBOOK)
th.save_json({'tables': TABLES, 'human': HUMAN}, P.run_file('final_tables.json'), CFG, NOTEBOOK)
print(json.dumps(th.strip_ci(PN), indent=1)[:4000])
print('Code_6 complete.')